In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# --- 1. Function to convert YOLO bounding box format to Faster R-CNN bbox format ---
def yolo_to_frcnn_bbox(x_center, y_center, width, height, img_width, img_height):
    """
    Convert bounding box from YOLO format (normalized center x,y and width, height)
    to Faster R-CNN format (absolute xmin, ymin, xmax, ymax coordinates).
    """
    xmin = (x_center - width / 2) * img_width
    ymin = (y_center - height / 2) * img_height
    xmax = (x_center + width / 2) * img_width
    ymax = (y_center + height / 2) * img_height
    return [xmin, ymin, xmax, ymax]

# --- 2. Function to load YOLO annotation files and convert them to Faster R-CNN format ---
def load_yolo_annotations(img_dir, label_dir, class_map):
    """
    Reads images and corresponding YOLO label files,
    converts bounding boxes to Faster R-CNN format,
    and maps class indices to Faster R-CNN compatible labels.

    Args:
        img_dir (str): directory path containing images
        label_dir (str): directory path containing YOLO label .txt files
        class_map (dict): dictionary mapping YOLO class indices to Faster R-CNN class indices (starting from 1)

    Returns:
        img_paths (list): list of image file paths with valid bounding boxes
        annotations (list): list of dicts containing 'boxes' and 'labels' for each image
    """
    img_files = sorted(os.listdir(img_dir))
    label_files = sorted(os.listdir(label_dir))

    img_paths = []
    annotations = []

    # Iterate through all image and label file pairs
    for img_file, label_file in zip(img_files, label_files):
        img_path = os.path.join(img_dir, img_file)
        label_path = os.path.join(label_dir, label_file)

        # Open image and get dimensions
        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size

        boxes = []
        labels = []

        # Read label file lines
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                class_idx = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])

                # Convert YOLO bbox to absolute bbox
                bbox = yolo_to_frcnn_bbox(x_center, y_center, width, height, img_width, img_height)
                boxes.append(bbox)
                labels.append(class_map[class_idx])

        # Skip images without any bounding boxes
        if len(boxes) == 0:
            continue

        # Store valid image path and annotations
        img_paths.append(img_path)
        annotations.append({
            'boxes': boxes,
            'labels': labels
        })

    return img_paths, annotations

# --- 3. Custom Dataset Class for Faster R-CNN ---
class CustomDataset(Dataset):
    """
    Dataset class compatible with PyTorch Faster R-CNN.
    Returns images and target dict with bounding boxes and labels.
    """
    def __init__(self, img_paths, annotations, transforms=None):
        self.img_paths = img_paths
        self.annotations = annotations
        self.transforms = transforms

    def __getitem__(self, idx):
        # Load image
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        # Load annotation for this image
        target = self.annotations[idx]

        # Convert boxes and labels to tensors
        boxes = torch.as_tensor(target['boxes'], dtype=torch.float32)
        labels = torch.as_tensor(target['labels'], dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels

        # Apply transformations if any
        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.img_paths)

# --- 4. Compose image transformations ---
def get_transform(train):
    """
    Compose image transforms.
    Includes conversion to tensor and random horizontal flip augmentation during training.
    """
    transforms = []
    transforms.append(T.ToTensor())
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    return T.Compose(transforms)

# --- 5. Prepare dataset paths and load data ---
train_img_dir = "/kaggle/input/fyufufuf/Data/Kangaroo-Koala.v14-kangaroo-koala-v13.yolov8/train/images"
train_label_dir = "/kaggle/input/fyufufuf/Data/Kangaroo-Koala.v14-kangaroo-koala-v13.yolov8/train/labels"

# Map YOLO class indices to Faster R-CNN class indices (starting at 1)
class_map = {0: 1, 1: 2}

# Load image paths and annotations
train_img_paths, train_annotations = load_yolo_annotations(train_img_dir, train_label_dir, class_map)

print(f"Loaded {len(train_img_paths)} images with annotations")

# Initialize the custom dataset
train_dataset = CustomDataset(train_img_paths, train_annotations, transforms=get_transform(train=True))

# --- 6. DataLoader ---
def collate_fn(batch):
    """
    Custom collate function to batch images and targets.
    """
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

# --- 7. Prepare Faster R-CNN Model ---
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load pre-trained Faster R-CNN model with ResNet50 backbone
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Number of classes = number of labels + background
num_classes = 3  # 2 classes + background

# Replace the classifier head with a new one (adjust output features)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Move model to device (GPU or CPU)
model.to(device)

# --- 8. Optimizer and Learning Rate Scheduler ---
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# --- 9. Training Loop ---
num_epochs = 5

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    total_loss = 0

    # Iterate over batches
    for i, (images, targets) in enumerate(train_loader):
        # Move images and targets to device
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass: compute losses
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backpropagation and optimizer step
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        # Print loss every 10 steps
        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i}/{len(train_loader)}], Loss: {losses.item():.4f}")

    # Step the learning rate scheduler after each epoch
    lr_scheduler.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}] finished. Avg Loss: {avg_loss:.4f}")


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Initialize the mean average precision (mAP) metric
map_metric = MeanAveragePrecision()

model.eval()  # Set the model to evaluation mode (disables dropout, batchnorm updates, etc.)
map_metric.reset()  # Reset the metric state before starting evaluation

with torch.no_grad():  # Disable gradient computations for evaluation (saves memory, speeds up)
    for images, targets in test_loader:
        # Move images to the appropriate device (GPU or CPU)
        images = list(img.to(device) for img in images)
        
        # Run the model to get predictions; outputs is a list of dicts, one per image
        outputs = model(images)

        preds = []
        gts = []
        # For each image in the batch, prepare prediction and ground truth in the format
        # required by torchmetrics MeanAveragePrecision
        for out, tgt in zip(outputs, targets):
            preds.append({
                "boxes": out["boxes"].cpu(),       # Predicted bounding boxes (tensor)
                "scores": out["scores"].cpu(),     # Confidence scores for each box
                "labels": out["labels"].cpu(),     # Predicted class labels
            })
            gts.append({
                "boxes": tgt["boxes"].cpu(),       # Ground truth bounding boxes
                "labels": tgt["labels"].cpu(),     # Ground truth class labels
            })

        # Update the metric state with the predictions and ground truths from this batch
        map_metric.update(preds, gts)

# After processing all batches, compute and get the final mAP result
result = map_metric.compute()

print(result)  # Prints a dict containing mAP and related detection metrics


In [ ]:
import numpy as np
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# --- Function to compute Intersection over Union (IoU) between two boxes ---
def iou(boxA, boxB):
    """
    Calculate Intersection over Union (IoU) of two bounding boxes.

    Args:
        boxA, boxB: lists or arrays of bounding box coordinates [xmin, ymin, xmax, ymax]

    Returns:
        IoU score (float)
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)  # Intersection area
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])  # Area of boxA
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])  # Area of boxB

    # Compute IoU
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)

# --- Function to compute Precision, Recall, and F1-score for one image ---
def compute_detection_metrics(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5, score_threshold=0.5):
    """
    Match predicted boxes with ground truth boxes using IoU and compute metrics.

    Args:
        pred_boxes: list of predicted bounding boxes [[xmin, ymin, xmax, ymax], ...]
        pred_scores: list of confidence scores for each predicted box
        gt_boxes: list of ground truth bounding boxes [[xmin, ymin, xmax, ymax], ...]
        iou_threshold: IoU threshold to consider a prediction correct
        score_threshold: minimum confidence score to consider a predicted box

    Returns:
        precision, recall, f1 score (floats)
    """
    # Filter out predicted boxes below confidence threshold
    filtered_preds = [(box, score) for box, score in zip(pred_boxes, pred_scores) if score >= score_threshold]
    pred_boxes = [b for b, s in filtered_preds]

    TP = 0  # True positives
    FP = 0  # False positives
    matched_gt = set()  # Keep track of ground truth boxes already matched

    # For each predicted box, check if it matches any ground truth box
    for pred_box in pred_boxes:
        matched = False
        for i, gt_box in enumerate(gt_boxes):
            if i in matched_gt:
                continue  # This ground truth box is already matched
            if iou(pred_box, gt_box) >= iou_threshold:
                TP += 1
                matched_gt.add(i)
                matched = True
                break
        if not matched:
            FP += 1  # Predicted box does not match any ground truth box

    FN = len(gt_boxes) - len(matched_gt)  # Ground truth boxes missed by predictions

    # Calculate precision, recall, f1 with small epsilon to avoid division by zero
    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)

    return precision, recall, f1

# --- Initialize mean Average Precision (mAP) metric from torchmetrics ---
map_metric = MeanAveragePrecision()

# Lists to store metrics for all test images
precisions = []
recalls = []
f1_scores = []

model.eval()  # Set model to evaluation mode
map_metric.reset()  # Reset metric before computing

with torch.no_grad():  # Disable gradient calculation for inference
    for images, targets in test_loader:
        # Move images to device (CPU or GPU)
        images = list(img.to(device) for img in images)
        outputs = model(images)  # Get predictions from model

        preds = []  # List to store batch predictions for mAP calculation
        gts = []    # List to store batch ground truths for mAP calculation

        for out, tgt in zip(outputs, targets):
            # Convert tensors to lists for custom metric computation
            pred_boxes = out['boxes'].cpu().numpy().tolist()
            pred_scores = out['scores'].cpu().numpy().tolist()
            gt_boxes = tgt['boxes'].cpu().numpy().tolist()

            # Compute per-image precision, recall, f1 using custom function
            p, r, f1 = compute_detection_metrics(pred_boxes, pred_scores, gt_boxes)

            # Append to lists to average later
            precisions.append(p)
            recalls.append(r)
            f1_scores.append(f1)

            # Prepare data for torchmetrics mAP computation
            preds.append({
                "boxes": out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu(),
            })
            gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
            })

        # Update mAP metric with current batch predictions and ground truths
        map_metric.update(preds, gts)

# Calculate mean metrics across all test images
mean_precision = np.mean(precisions)
mean_recall = np.mean(recalls)
mean_f1 = np.mean(f1_scores)

# Compute final mean Average Precision (mAP) across all batches
mean_ap = map_metric.compute()['map'].item()

# Print averaged results
print(f"Average Precision on test set: {mean_precision:.4f}")
print(f"Average Recall on test set: {mean_recall:.4f}")
print(f"Average F1-score on test set: {mean_f1:.4f}")
print(f"Mean Average Precision (mAP) on test set: {mean_ap:.4f}")


In [ ]:
empty_bbox_indices = []  # List to store indices of images without bounding boxes

# Iterate through all training annotations
for i, ann in enumerate(train_annotations):
    # Check if the image has zero bounding boxes
    if len(ann['boxes']) == 0:
        empty_bbox_indices.append(i)  # Save index if no bbox

# Print how many images have no bounding boxes
print(f"There are {len(empty_bbox_indices)} images without bounding boxes.")

# Print the file paths of images that have no bounding boxes
for idx in empty_bbox_indices:
    print(f"Image {idx}: {train_img_paths[idx]}")


In [ ]:
filtered_img_paths = []       # List to store image paths after filtering
filtered_annotations = []     # List to store annotations after filtering

# Loop through all training images
for i in range(len(train_img_paths)):
    # Skip images that have no bounding boxes (indices in empty_bbox_indices)
    if i not in empty_bbox_indices:
        filtered_img_paths.append(train_img_paths[i])       # Keep valid image path
        filtered_annotations.append(train_annotations[i])   # Keep valid annotation

# Print number of images left after filtering
print(f"Number of images after filtering: {len(filtered_img_paths)}")

# Recreate dataset and dataloader with filtered images and annotations
train_dataset = CustomDataset(filtered_img_paths, filtered_annotations, transforms=get_transform(train=True))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
